In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.gru(x); out = self.dropout(out[:, -1, :]); out = self.fc(out)
        return out

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.lstm(x); out = self.dropout(out[:, -1, :]); out = self.fc(out)
        return out

In [ ]:
def create_sequences(data, n_steps, target_col):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data.iloc[i:(i + n_steps)].values)
        y.append(data.iloc[i + n_steps][target_col])
    return np.array(X), np.array(y).reshape(-1, 1)

In [ ]:
def calculate_mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    nonzero_mask = actual != 0
    if not np.any(nonzero_mask): return float('inf')
    return np.mean(np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])) * 100

In [ ]:
def get_predictions(model, loader, device):
    model.eval()
    all_predictions = []
    with torch.no_grad():
        for batch_X, _ in loader:
            outputs = model(batch_X.to(device))
            all_predictions.append(outputs.cpu().numpy())
    return np.concatenate(all_predictions)

In [ ]:
def inverse_transform_predictions(predictions, original_df, target_col_name, scaler_obj):
    price_col_index = original_df.columns.get_loc(target_col_name)
    dummy_array = np.zeros((len(predictions), original_df.shape[1]))
    dummy_array[:, price_col_index] = predictions.flatten()
    return scaler_obj.inverse_transform(dummy_array)[:, price_col_index]

In [ ]:
CROP_NAME = 'Turmeric'
CROP_CSV_PATH = f'../Weather_Merged_CSVs/{CROP_NAME}.csv'

In [ ]:
GRU_MODEL_PATH = f'GRU_Results/{CROP_NAME}_best_model_gru.pth'
GRU_OPTIMIZED_MODEL_PATH = f'GRU_Results_Optimized/{CROP_NAME}_best_model_gru_optimized.pth'
LSTM_MODEL_PATH = f'LSTM_Results/{CROP_NAME}_best_model_lstm.pth'
LSTM_OPTIMIZED_MODEL_PATH = f'LSTM_Results_Optimized/{CROP_NAME}_best_model_lstm_optimized.pth'

In [ ]:
STANDARD_N_STEPS = 14 
print(f"--- Starting Ensemble Analysis for: {CROP_NAME} with n_steps = {STANDARD_N_STEPS} ---")

In [ ]:
df_original = pd.read_csv(CROP_CSV_PATH)
df_original['Price Date'] = pd.to_datetime(df_original['Price Date'])

# Create a copy for processing
df_processed = df_original.copy()
df_processed.set_index('Price Date', inplace=True); df_processed.sort_index(inplace=True)

# Feature Engineering & Scaling
df_processed['day_of_year'] = df_processed.index.dayofyear; df_processed['week_of_year'] = df_processed.index.isocalendar().week.astype(int); df_processed['month'] = df_processed.index.month
categorical_cols = ['District Name', 'Market Name', 'Commodity', 'Variety', 'Grade']
for col in categorical_cols:
    if df_processed[col].dtype == 'object': df_processed[col] = LabelEncoder().fit_transform(df_processed[col])
scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = pd.DataFrame(scaler.fit_transform(df_processed), columns=df_processed.columns)
target_column = 'Modal Price (Rs./Quintal)'

In [ ]:
X, y = create_sequences(df_scaled, STANDARD_N_STEPS, target_column)

# Create the test set
train_val_split = int(0.8 * len(X))
_, X_test = X[:train_val_split], X[train_val_split:]
_, y_test = y[:train_val_split], y[train_val_split:]
X_test_tensor = torch.from_numpy(X_test).float(); y_test_tensor = torch.from_numpy(y_test).float()
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)
print(f"Test data created with {len(X_test)} samples.")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
input_size = X_test.shape[2]
model_params = {'input_size': input_size, 'hidden_size': 50, 'num_layers': 2, 'output_size': 1, 'dropout_prob': 0.2}
models_to_load = {
    "GRU General": (GRUModel, GRU_MODEL_PATH), "GRU Optimized": (GRUModel, GRU_OPTIMIZED_MODEL_PATH),
    "LSTM General": (LSTMModel, LSTM_MODEL_PATH), "LSTM Optimized": (LSTMModel, LSTM_OPTIMIZED_MODEL_PATH)
}

In [ ]:
predictions_dict = {}
for name, (ModelClass, path) in models_to_load.items():
    model = ModelClass(**model_params).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    scaled_preds = get_predictions(model, test_loader, device)
    predictions_dict[name] = inverse_transform_predictions(scaled_preds, df_scaled, target_column, scaler)

In [ ]:
actual_prices = inverse_transform_predictions(y_test, df_scaled, target_column, scaler)
ensemble_predictions = np.mean([preds for preds in predictions_dict.values()], axis=0)
predictions_dict["ENSEMBLE (Average)"] = ensemble_predictions

In [ ]:
print("\nCreating detailed results file with district information...")
# This is the key step. We find the original data that corresponds to the test set.
test_set_start_index = train_val_split + STANDARD_N_STEPS
final_original_data = df_original.iloc[test_set_start_index:].copy()

# Check for length consistency
if len(final_original_data) != len(actual_prices):
    # This can happen if the last few rows don't form a full sequence
    final_original_data = final_original_data.iloc[:len(actual_prices)]

# Create the detailed results DataFrame
final_predictions_df = pd.DataFrame({
    'Price_Date': final_original_data['Price Date'],
    'District_Name': final_original_data['District Name'],
    'Market_Name': final_original_data['Market Name'],
    'Actual_Price': actual_prices
})

# Add all model predictions
for name, preds in predictions_dict.items():
    final_predictions_df[f"{name.replace(' ', '_')}_Pred"] = preds

output_filename = f"./Ensemble-District-Wise/{CROP_NAME}_district_level_predictions.csv"
final_predictions_df.to_csv(output_filename, index=False)
print(f"Detailed predictions with district info saved to '{output_filename}'")

In [ ]:
evaluation_results = []
for name, predictions in predictions_dict.items():
    rmse = np.sqrt(mean_squared_error(actual_prices, predictions))
    mape = calculate_mape(actual_prices, predictions)
    evaluation_results.append({"Model": name, "RMSE": rmse, "MAPE (%)": mape})
results_summary_df = pd.DataFrame(evaluation_results).sort_values(by="MAPE (%)")
print("\n--- Overall Performance Comparison ---")
print(results_summary_df.to_string(index=False))